# Cardiac MRI Analysis & Diagnosis (ACDC)
First: **Runtime → Change runtime type → T4 GPU**. Run cells top to bottom.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PROJECT = '/content/drive/MyDrive/cardiac_project'
%cd $PROJECT
!ls

In [ ]:
!pip install -q -r requirements.txt

## Get the data (pick ONE option)
**A (official):** download from the ACDC / humanheart-project site, upload the zip to `cardiac_project/data/` on Drive, then unzip below.

**B (Hugging Face mirror, third-party copy):** run the second cell. It skips the large `_4d` files, which this project does not use.

In [ ]:
# Option A - change the zip name to yours
!unzip -q -o data/YOUR_ACDC_ZIP.zip -d data/

In [ ]:
# Option B
from huggingface_hub import snapshot_download
snapshot_download(repo_id='msepulvedagodoy/acdc', repo_type='dataset',
                  local_dir='data/acdc', ignore_patterns=['*_4d.nii.gz'])

## Step 0 — inspect the dataset (must say ALL CHECKS PASSED)

In [ ]:
!python src/00_inspect_dataset.py

In [ ]:
from IPython.display import Image
Image('outputs/figures/example_patient001.png')

In [ ]:
!python src/01_preprocess.py

In [ ]:
!python src/02_train_unet.py --arch unet --epochs 40

In [ ]:
# optional: U-Net++ for comparison (then use --arch unetpp in step 3 if it is better)
# !python src/02_train_unet.py --arch unetpp --epochs 40

In [ ]:
!python src/03_eval_segmentation.py --arch unet

In [ ]:
!python src/04_extract_features.py

In [ ]:
!python src/05_train_xgboost.py

In [ ]:
!python src/06_train_cnn.py --epochs 25

In [ ]:
!python src/07_compare_models.py

In [ ]:
!python src/08_explainability.py

In [ ]:
from IPython.display import Image, display
for f in ['seg_examples_unet','confusion_matrices','shap_global','gradcam_examples']:
    display(Image(f'outputs/figures/{f}.png'))

## Step 9 - metrics with 95% CI, then build the paper files

In [ ]:
!python src/09_report_metrics.py --boot 2000

In [ ]:
!pip install -q python-docx python-pptx
!python paper/make_diagrams.py
!python paper/build_docs.py

## Dashboard — open the gradio.live link it prints. Stop the cell to close it.

In [ ]:
!python src/app.py